## CODE GENERATOR

__The requirement:__ Use a Frontier model to generate high performance C++ code from Python code

In [3]:
!uv pip install --quiet openai

In [9]:
# imports

import os
import subprocess
from rich import print
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [6]:
load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b"
groq_base_url = os.getenv("GROQ_BASE_URL")

if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")

GROQ API Key exists and begins gsk_GIUx


In [8]:
# Connect to groq client

groq = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

#### PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [11]:
!uv pip install --quiet system_info

In [29]:
from system_info import sysinfo

# 1. Check what functions are hidden inside the actual sub-module
print(dir(sysinfo))

print("System Info Module Version:", sysinfo.sysInfo.items())

[
    'RAM_Types',
    'SystemInfo',
    '__builtins__',
    '__cached__',
    '__doc__',
    '__file__',
    '__loader__',
    '__name__',
    '__package__',
    '__spec__',
    '__warningregistry__',
    'cpuinfo',
    'math',
    'os',
    'os_info',
    'platform',
    'psutil',
    'shutil',
    'sk',
    'subprocess',
    'sys',
    'sysInfo'
]

System Info Module Version: dict_items([('Processor', 'AMD Ryzen 7 250 w/ Radeon 780M Graphics'), ('CPU', 16), 
('Ip', '10.238.19.152'), ('OS Version', '10.0.26200'), ('Total Disk Space', '951 GB'), ('HD Size', '951 GB'), 
('Available Space', '621 GB'), ('HD_Type', None), ('Operating System', 'Microsoft Windows 11 Home Single 
Language'), ('Host Name', 'LAPTOP-NJTCAHH8'), ('CPU_Core', None), ('Manufacturer', None), ('Model', None), 
('Ram_Type', None), ('Ram_Size', '15 GB'), ('Serial_Number', None)])

In [31]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{sysinfo.sysInfo.items()}
"""

response = groq.chat.completions.create(model=groq_model, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

**Short answer:**  
You probably don’t have a C++ compiler installed yet (most Windows 11 Home machines ship only the Windows SDK).  
If you do, you’ll see a message like `g++ is not recognized` or `cl is not recognized` when you try the commands below.  
The easiest path to a ready‑to‑compile environment on Windows is to install **MinGW‑w64** (the GCC compiler suite) and add it to your system PATH.

---

## 1.  Check whether you already have a compiler

Open a PowerShell or CMD window and run:

```powershell
g++ --version
# or
cl
```

* If you see a version string, you’re already set up.  
* If you get `command not found / is not recognized as an internal or external command…`, you need to install one.

---

## 2.  Install a C++ compiler (MinGW‑w64)

The following steps install the *stand‑alone* MinGW‑w64 toolchain, which works great for single‑file builds.

1. **Download the installer**  
   Go to the official MinGW‑w64 download page:  
   <https://mingw-w64.org/doku.php/download/mingw-builds>  

   Pick the *x86_64* architecture, the *posix* threads, and the *seh* exception model (the most common choice).  
   Click the “**mingw-w64-install.exe**” link to download the installer (≈ 30 MB).

2. **Run the installer**  
   * Choose an install location like `C:\mingw-w64`.
   * Make sure the *“Add to PATH”* option is ticked (if not, you’ll have to add it manually later).

3. **Verify the installation**  
   Open a **new** PowerShell window (so it picks up the updated PATH) and run:

   ```powershell
   g++ --version
   ```

   You should see something like:

   ```
   g++ (x86_64-posix-seh) 12.1.0
   ```

   If you still don’t see it, add the bin directory manually:

   ```powershell
   $env:Path += ";C:\mingw-w64\mingw64\bin"
   ```

   (Replace the path with the actual install location.)

4. **Optional – Install `make` (not required for a single file)**  
   If you ever need `make`, install it via the same installer by selecting the *make* component, or simply install [Make for Windows](https://gnuwin32.sourceforge.net/packages/make.htm) and add its `bin` to the PATH.

---

## 3.  Compile and run a single `main.cpp` file from Python

Below is a minimal Python snippet that:

1. **Compiles** `main.cpp` to `main.exe` with *maximum performance* (`-O3`, `-march=native`, `-DNDEBUG`).  
2. **Runs** the resulting executable.  
3. **Returns** its stdout.

> **NOTE**: If your source file uses C++20 features, replace `-std=c++17` with `-std=c++20`.

```python
import subprocess
import os

# Make sure we’re in the directory that contains main.cpp
os.chdir(r"C:\path\to\your\source")   # <-- edit this line

# 1) Compile
compile_command = [
    "g++",                # the compiler
    "-O3",                # optimize for speed
    "-march=native",      # use CPU‑specific instructions
    "-DNDEBUG",           # disable debug checks
    "-std=c++17",         # language standard (change if needed)
    "main.cpp",           # source file
    "-o", "main.exe"      # output binary name
]

compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True
)

print("Compilation succeeded:")
print(compile_result.stdout)
print(compile_result.stderr)   # usually empty for a clean build

# 2) Run
run_command = ["./main.exe"]  # or simply ["main.exe"] on Windows

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)

print("Program output:")
print(run_result.stdout)
```

### Quick explanation

| Step | What the command does | Why it’s chosen |
|------|-----------------------|-----------------|
| `g++ … -O3 -march=native -DNDEBUG -std=c++17 main.cpp -o main.exe` | Compiles with full optimization, CPU‑specific code generation, and no debug assertions. | Gives the fastest binary on your machine. |
| `./main.exe` | Executes the produced binary. | Standard on Windows for an executable in the current directory. |

---

## 4.  What if you already have a compiler?

If `g++ --version` or `cl` returned a version string, you can skip the install steps and just use the corresponding compiler command.  
For example, with **MSVC** (`cl.exe`):

```python
compile_command = [
    "cl",
    "/O2",          # max speed
    "/EHsc",        # enable C++ exceptions
    "/std:c++17",
    "main.cpp",
    "/Fe:main.exe"
]
run_command = ["main.exe"]
```

Just replace the `compile_command` and `run_command` lists with the one that matches your installed compiler.

---

### TL;DR

1. **Check**: `g++ --version` or `cl`.  
2. **If missing** → Install MinGW‑w64, add to PATH.  
3. **Use** the Python snippet above (or the MSVC‑specific snippet if you have Visual Studio).  
4. Run the script – it will compile `main.cpp` and print its output.

Happy coding!

#### If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

#### And now, on with the main task

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{sysinfo.sysInfo.items()}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [ ]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

In [ ]:
port(groq, groq_model, pi)

### Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()